# Demo 2: Cognitive Loop (วงจรการคิดของ Agent)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tiraphap/Gen-Agent-Lectures-at-Econ-TU/blob/master/teaching/demos/demo_cognitive_loop.ipynb)

---

## Overview

สคริปต์นี้สาธิต **5 ขั้นตอนของ Cognitive Loop** ใน Generative Agents:

```
    ┌─────────────────────────────────────────────────┐
    │              Cognitive Loop (1 tick)             │
    │                                                 │
    │   1. PERCEIVE (รับรู้)    → สังเกตสิ่งรอบตัว     │
    │   2. RETRIEVE (ดึง)      → ดึงความทรงจำที่เกี่ยวข้อง │
    │   3. PLAN (วางแผน)       → [LLM] คิดว่าจะทำอะไร  │
    │   4. EXECUTE (ลงมือทำ)    → ทำตามแผน              │
    │   5. REFLECT (สะท้อน)    → [LLM] สร้างความทรงจำใหม่ │
    │                                                 │
    └─────────────────────────────────────────────────┘
```

**หลักการสำคัญ:**
- ทุกการตัดสินใจผ่าน LLM (ไม่ใช่ if-else)
- Agent มี "บุคลิก" ที่ส่งผลต่อการตัดสินใจ
- มี toggle เปิด/ปิดการใช้ LLM จริง

| Step | What it does | Uses LLM? |
|------|-------------|-----------|
| PERCEIVE | Agent สังเกตสิ่งรอบตัว | No |
| RETRIEVE | ดึงความทรงจำที่เกี่ยวข้อง | No |
| PLAN | ใช้ LLM คิดว่าจะทำอะไร | **Yes** |
| EXECUTE | ทำตามแผน | No |
| REFLECT | สร้างความทรงจำใหม่ | **Yes** |

---
**ผศ.ดร.ถิรภาพ ฟักทอง** | คณะเศรษฐศาสตร์ มหาวิทยาลัยธรรมศาสตร์

In [ ]:
# ============================================================
# Config: เปิด/ปิดการใช้ LLM จริง
# ============================================================
# False = ใช้ข้อมูล mock (ไม่ต้องมี API key) <- เหมาะสำหรับ demo ในห้องเรียน
# True  = เรียก OpenAI API จริง (ต้องมี OPENAI_API_KEY)
USE_REAL_LLM = False

# ============================================================
# Imports
# ============================================================
import json
import os
import math
from datetime import datetime, timedelta
from typing import List, Dict, Any, Optional

print(f"โหมดปัจจุบัน: {'LLM จริง (OpenAI API)' if USE_REAL_LLM else 'Mock (ข้อมูลจำลอง) - ไม่ต้องใช้ API key'}")

In [ ]:
# ============================================================
# Helper Functions (ไม่มี ANSI colors - ใช้ print ธรรมดา)
# ============================================================

def print_header(text: str):
    """แสดงหัวข้อหลักแบบมีกรอบ"""
    width = 64
    print(f"\n{'=' * width}")
    print(f"  {text}")
    print(f"{'=' * width}")


def print_step_header(step_num: int, step_name: str, thai_desc: str,
                      uses_llm: bool = False):
    """แสดงหัวข้อของแต่ละ step ใน cognitive loop"""
    llm_badge = " [ใช้ LLM]" if uses_llm else ""
    print(f"\n{'─' * 64}")
    print(f"  Step {step_num}: {step_name} - {thai_desc}{llm_badge}")
    print(f"{'─' * 64}")


def print_box(title: str, content: str):
    """แสดงกล่องข้อความ"""
    lines = content.strip().split('\n')
    max_len = max(len(line) for line in lines)
    width = max(max_len + 4, len(title) + 4)

    print(f"  ┌{'─' * width}┐")
    print(f"  │ {title}{' ' * (width - len(title) - 1)}│")
    print(f"  ├{'─' * width}┤")
    for line in lines:
        padding = width - len(line) - 1
        print(f"  │ {line}{' ' * padding}│")
    print(f"  └{'─' * width}┘")


def make_bar(value: float, max_width: int = 20) -> str:
    """สร้าง bar chart ด้วย Unicode blocks (ไม่มีสี)"""
    filled = int(value * max_width)
    bar = "█" * filled + "░" * (max_width - filled)
    return bar


print("Helper functions loaded.")

In [ ]:
# ============================================================
# ข้อมูล Agent: Customer Profile + Stores
# ============================================================

def load_customer_data() -> Dict:
    """โหลดข้อมูลลูกค้าจากไฟล์ data/ หรือใช้ข้อมูลที่ฝังไว้ถ้าไม่พบไฟล์

    พยายามโหลดจาก ../../data/customers.json ก่อน
    ถ้าไม่พบ จะใช้ข้อมูลที่ฝังไว้ในไฟล์นี้ (fallback)
    """
    data_paths = [
        os.path.join(os.getcwd(), "..", "..", "data", "customers.json"),
        os.path.join(os.getcwd(), "data", "customers.json"),
    ]

    for path in data_paths:
        try:
            abs_path = os.path.abspath(path)
            with open(abs_path, "r", encoding="utf-8") as f:
                data = json.load(f)
                for customer in data.get("customers", []):
                    if customer["id"] == "cust_001":
                        print(f"  โหลดข้อมูลจาก: {abs_path}")
                        return customer
        except (FileNotFoundError, json.JSONDecodeError, KeyError):
            continue

    # ข้อมูล fallback ถ้าโหลดไฟล์ไม่ได้
    print("  ใช้ข้อมูล fallback (ไม่พบไฟล์ customers.json)")
    return {
        "id": "cust_001",
        "name": "สมศรี",
        "age": 45,
        "gender": "female",
        "job": "แม่บ้าน",
        "location": {"lat": 13.81, "lng": 100.55},
        "financial": {
            "monthly_income": 35000,
            "living_situation": "พอมีพอใช้ เก็บออมได้บ้าง",
            "financial_goal": "เก็บเงินให้ลูกเรียนมหาลัย"
        },
        "household": {"size": 4, "type": "บ้านเดี่ยว อยู่กับสามีและลูก 2 คน"},
        "personality": {
            "price_sensitivity": 0.8,
            "brand_loyalty": 0.6,
            "quality_preference": 0.5,
            "convenience_preference": 0.4,
            "promotion_attraction": 0.9,
            "impulse_buying": 0.3
        },
        "innate_traits": ["ใจเย็น", "ละเอียดรอบคอบ", "ชอบเปรียบเทียบราคา"],
        "interests": {
            "hobbies": ["ทำอาหาร", "ดูละคร", "ปลูกต้นไม้"],
            "favorite_categories": ["food", "household", "fresh_produce"]
        },
        "routines": [
            "ดูใบปลิวโปรโมชั่นทุกสัปดาห์",
            "ไปซื้อของสดตลาดเช้าวันอาทิตย์",
            "ทำกับข้าวให้ครอบครัวทุกวัน"
        ],
        "pain_points": [
            "ไม่ชอบร้านที่จอดรถยาก",
            "หงุดหงิดถ้าของที่ต้องการหมด",
            "ไม่ชอบพนักงานไม่สุภาพ"
        ],
        "shopping_patterns": {
            "frequency": "weekly",
            "avg_spend": 2500,
            "preferred_days": ["saturday", "sunday"],
            "preferred_times": ["morning", "afternoon"],
            "preferred_stores": ["CJ More", "Big C"]
        },
        "background": "แม่บ้านที่ดูแลครอบครัว 4 คน ชอบดูใบปลิวโปรโมชั่นทุกสัปดาห์"
    }


def get_household_state() -> Dict:
    """สถานะของบ้าน (inventory) ณ ปัจจุบัน"""
    data_paths = [
        os.path.join(os.getcwd(), "..", "..", "data", "household_states.json"),
    ]
    for path in data_paths:
        try:
            abs_path = os.path.abspath(path)
            with open(abs_path, "r", encoding="utf-8") as f:
                data = json.load(f)
                for state in data.get("household_states", []):
                    if state["customer_id"] == "cust_001":
                        return state
        except (FileNotFoundError, json.JSONDecodeError, KeyError):
            continue

    return {
        "customer_id": "cust_001",
        "inventory": {
            "milk": {"current": 1.0, "unit": "carton", "threshold": 1.0,
                     "consumption_per_day": 0.5, "category": "dairy"},
            "eggs": {"current": 4, "unit": "pieces", "threshold": 4,
                     "consumption_per_day": 3, "category": "food"},
            "rice": {"current": 4.0, "unit": "kg", "threshold": 2.0,
                     "consumption_per_day": 0.4, "category": "food"},
            "laundry_detergent": {"current": 0.3, "unit": "bottle", "threshold": 0.3,
                                  "consumption_per_day": 0.03, "category": "household"},
        },
        "shopping_list": [
            {"item": "milk", "urgency": "high", "reason": "เหลือแค่ 1 กล่อง ถึง threshold แล้ว"},
            {"item": "eggs", "urgency": "high", "reason": "เหลือ 4 ฟอง ถึง threshold แล้ว"},
            {"item": "laundry_detergent", "urgency": "medium", "reason": "สามีฝากซื้อ ใกล้หมด"}
        ],
        "last_shopping_date": "2024-01-06",
        "monthly_budget": 10000,
        "budget_remaining": 7500
    }


def get_environment_data() -> Dict:
    """ข้อมูลสภาพแวดล้อม (ร้านค้า, โปรโมชั่น)"""
    return {
        "current_day": "Saturday",
        "current_time": "09:00",
        "nearby_stores": [
            {
                "name": "CJ More",
                "distance_km": 2.5,
                "current_promotions": [
                    "นมสด Dutch Mill ลด 30% (เฉพาะสมาชิก)",
                    "ไข่ไก่ CP ซื้อ 2 แถม 1",
                    "ผงซักฟอก Attack ลด 25% เมื่อซื้อ 2 ชิ้น"
                ],
                "features": ["parking", "food_court", "pharmacy"],
                "avg_price_level": "ปานกลาง"
            },
            {
                "name": "Big C สาขารัชดา",
                "distance_km": 4.0,
                "current_promotions": [
                    "นม Meiji ซื้อ 2 แถม 1",
                    "กระดาษทิชชู่ ลด 20% สมาชิก"
                ],
                "features": ["parking", "food_court"],
                "avg_price_level": "ปานกลาง"
            },
            {
                "name": "7-Eleven สาขาหมู่บ้าน",
                "distance_km": 0.3,
                "current_promotions": [],
                "features": ["24hr"],
                "avg_price_level": "สูง"
            },
            {
                "name": "Tops สาขา Central",
                "distance_km": 3.0,
                "current_promotions": [
                    "ผักออร์แกนิค ลด 15%"
                ],
                "features": ["premium", "organic"],
                "avg_price_level": "สูง"
            }
        ]
    }


def get_relevant_memories() -> List[Dict]:
    """ความทรงจำที่เกี่ยวข้อง (จำลองผลจาก Memory Stream retrieval)"""
    return [
        {
            "description": "เปิดแอป CJ More เห็นว่าวันนี้มีโปรพิเศษนมสด ลด 30% เฉพาะสมาชิก",
            "recency": 0.990, "importance": 0.900, "relevance": 1.000,
            "total_score": 0.891, "hours_ago": 1, "type": "observation"
        },
        {
            "description": "วางแผนจะไปซื้อนมและไข่วันเสาร์เช้านี้ ต้องดูว่าร้านไหนโปรดีที่สุด",
            "recency": 0.941, "importance": 0.800, "relevance": 0.750,
            "total_score": 0.565, "hours_ago": 6, "type": "plan"
        },
        {
            "description": "ดูใบปลิว CJ More เห็นโปรไข่ไก่ ซื้อ 2 แถม 1 และผงซักฟอกลด 25%",
            "recency": 0.785, "importance": 0.700, "relevance": 0.500,
            "total_score": 0.275, "hours_ago": 24, "type": "observation"
        },
        {
            "description": "สรุปว่า Big C ราคาดีแต่ที่จอดรถยากมาก โดยเฉพาะวันเสาร์",
            "recency": 0.484, "importance": 0.700, "relevance": 0.333,
            "total_score": 0.113, "hours_ago": 72, "type": "reflection"
        },
        {
            "description": "พนักงาน CJ More บริการดีมาก ช่วยหิ้วของไปที่รถให้",
            "recency": 0.185, "importance": 0.600, "relevance": 0.250,
            "total_score": 0.028, "hours_ago": 168, "type": "observation"
        }
    ]


# โหลดข้อมูลทั้งหมด
customer = load_customer_data()
household = get_household_state()
environment = get_environment_data()

print(f"\nAgent: {customer['name']} ({customer['job']}, อายุ {customer['age']})")
print(f"Household: {customer['household']['size']} คน")
print(f"Budget remaining: {household['budget_remaining']:,} / {household['monthly_budget']:,} บาท")

In [ ]:
# ============================================================
# Mock LLM Responses
# ============================================================
# ข้อมูลจำลองนี้ใช้เมื่อ USE_REAL_LLM = False
# ออกแบบให้สมจริง เหมือนสิ่งที่ LLM จะตอบจริงๆ
# สังเกตว่า thinking_steps แสดงกระบวนการคิดทีละขั้น (Chain-of-Thought)

MOCK_DECISION = {
    "thinking_steps": [
        "วันนี้วันเสาร์เช้า ซึ่งเป็นวันที่ฉันชอบไปซื้อของ ดีเลย",
        "ดูจาก shopping list: นมเหลือแค่ 1 กล่อง (threshold=1.0) ต้องซื้อแน่ๆ ไข่ก็เหลือน้อย ผงซักฟอกสามีก็ฝากมา",
        "จากความทรงจำ: CJ More มีโปรนมสด ลด 30% เฉพาะสมาชิก เพิ่งเห็นในแอปเมื่อชั่วโมงที่แล้ว น่าสนใจมาก!",
        "CJ More ยังมีโปรไข่ ซื้อ 2 แถม 1 และผงซักฟอกลด 25% ด้วย ครบทุกอย่างที่ต้องซื้อเลย",
        "จำได้ว่า Big C จอดรถยากมากวันเสาร์ ไปทีก่อนหงุดหงิดมาก... CJ More บริการดีกว่า พนักงานช่วยหิ้วของด้วย",
        "ฉันเป็นคนดูราคาเป็นหลัก (price_sensitivity=0.8) และชอบโปร (promotion_attraction=0.9) CJ More มีโปรตรงกับที่ต้องซื้อทุกอย่าง",
        "ตัดสินใจ: ไป CJ More เช้านี้เลย! ซื้อนม ไข่ ผงซักฟอก ครบในที่เดียว ราคาดี บริการดี จอดรถง่าย"
    ],
    "decision": {
        "action": "go_shopping",
        "store": "CJ More",
        "items_to_buy": ["นมสด Dutch Mill (โปรลด 30%)", "ไข่ไก่ (ซื้อ 2 แถม 1)", "ผงซักฟอก (ลด 25%)"],
        "estimated_spend": 350,
        "reasoning": "เลือก CJ More เพราะมีโปรตรงกับสินค้าที่ต้องซื้อทุกอย่าง (นม ไข่ ผงซักฟอก) "
                     "ราคาคุ้มค่า บริการดี และจอดรถสะดวกกว่า Big C ในวันเสาร์"
    }
}

MOCK_REFLECTION = {
    "reflection": "การไปซื้อของที่ CJ More วันนี้คุ้มค่ามาก ได้โปรนมลด 30% ไข่ซื้อ 2 แถม 1 "
                  "และผงซักฟอกลด 25% รวมประหยัดได้เกือบ 150 บาท บริการก็ดี จอดรถง่าย "
                  "ครั้งหน้าถ้า CJ More มีโปรอีก ควรมาที่นี่ก่อนเลย",
    "importance": 7,
    "new_memory": "ซื้อของที่ CJ More ได้โปรดีมาก ประหยัดได้เกือบ 150 บาท บริการดี จอดรถสะดวก"
}


# ============================================================
# LLM Integration Functions
# ============================================================

def build_llm_prompt(customer: Dict, household: Dict, environment: Dict,
                     memories: List[Dict]) -> str:
    """สร้าง prompt สำหรับส่งให้ LLM ตัดสินใจ

    prompt นี้ประกอบด้วย 10 ส่วน ตาม spec:
    1. Identity (ตัวตน)
    2. Financial (การเงิน)
    3. Personality (บุคลิก)
    4. Innate Traits (ลักษณะนิสัย)
    5. Interests (ความสนใจ)
    6. Routines (กิจวัตร)
    7. Pain Points (สิ่งที่ไม่ชอบ)
    8. Relevant Memories (ความทรงจำที่เกี่ยวข้อง)
    9. Current Situation (สถานการณ์ปัจจุบัน)
    10. Question (คำถาม)
    """
    # แปลง personality scores เป็นคำอธิบาย
    p = customer["personality"]
    personality_desc = []
    trait_map = {
        "price_sensitivity": ("ความอ่อนไหวด้านราคา", "ไม่สนราคา", "ดูราคาเป็นหลัก"),
        "brand_loyalty": ("ความภักดีต่อแบรนด์", "เปลี่ยนแบรนด์ง่าย", "ภักดีมาก"),
        "quality_preference": ("ความใส่ใจคุณภาพ", "ถูกก็ได้", "ต้องคุณภาพดี"),
        "convenience_preference": ("ความสำคัญของความสะดวก", "ไปไกลได้", "ต้องสะดวก"),
        "promotion_attraction": ("ความสนใจโปรโมชั่น", "ไม่สนโปร", "ชอบดูโปรมาก"),
        "impulse_buying": ("การซื้อตามอารมณ์", "ซื้อตามแผนเท่านั้น", "ซื้อตามใจบ่อย"),
    }
    for key, (name, low_desc, high_desc) in trait_map.items():
        score = p.get(key, 0.5)
        desc = high_desc if score >= 0.6 else low_desc if score <= 0.4 else "ปานกลาง"
        personality_desc.append(f"  - {name}: {score:.1f}/1.0 ({desc})")

    # สร้าง memory text
    memory_text = ""
    for i, mem in enumerate(memories, 1):
        memory_text += (f"  {i}. [{mem['type']}] \"{mem['description']}\" "
                       f"(score: {mem['total_score']:.3f})\n")

    # สร้าง shopping list text
    shopping_list_text = ""
    for item in household.get("shopping_list", []):
        shopping_list_text += f"  - {item['item']}: {item['reason']} (ความเร่งด่วน: {item['urgency']})\n"

    # สร้าง store text
    store_text = ""
    for store in environment.get("nearby_stores", []):
        promos = ", ".join(store["current_promotions"]) if store["current_promotions"] else "ไม่มีโปร"
        store_text += (f"  - {store['name']} ({store['distance_km']} km) "
                      f"ราคา: {store['avg_price_level']} | โปร: {promos}\n")

    prompt = f"""คุณคือ {customer['name']} ({customer['job']}, อายุ {customer['age']} ปี)
{customer.get('background', '')}

## ข้อมูลครัวเรือน
- สมาชิก: {customer['household']['size']} คน ({customer['household']['type']})
- รายได้/เดือน: {customer['financial']['monthly_income']:,} บาท
- สถานะการเงิน: {customer['financial']['living_situation']}
- เป้าหมายการเงิน: {customer['financial']['financial_goal']}

## บุคลิกภาพ (Personality Traits)
{chr(10).join(personality_desc)}

## ลักษณะนิสัย
{', '.join(customer.get('innate_traits', []))}

## ความสนใจ
- งานอดิเรก: {', '.join(customer.get('interests', {}).get('hobbies', []))}
- หมวดสินค้าที่ชอบ: {', '.join(customer.get('interests', {}).get('favorite_categories', []))}

## กิจวัตร
{chr(10).join('- ' + r for r in customer.get('routines', []))}

## สิ่งที่ไม่ชอบ (Pain Points)
{chr(10).join('- ' + pp for pp in customer.get('pain_points', []))}

## ความทรงจำที่เกี่ยวข้อง (เรียงตามคะแนนจากมากไปน้อย)
{memory_text}
## สถานการณ์ปัจจุบัน
- วัน: {environment['current_day']}
- เวลา: {environment['current_time']}
- งบเหลือ: {household['budget_remaining']:,} บาท (จากทั้งหมด {household['monthly_budget']:,} บาท/เดือน)
- ซื้อของครั้งล่าสุด: {household['last_shopping_date']}

## รายการที่ต้องซื้อ (Shopping List)
{shopping_list_text}
## ร้านค้าใกล้เคียง
{store_text}
## คำถาม
จากข้อมูลทั้งหมดข้างต้น โปรดคิดทีละขั้นตอน (step-by-step) แล้วตัดสินใจว่า:
1. วันนี้จะไปซื้อของหรือไม่?
2. ถ้าไป จะไปร้านไหน?
3. จะซื้ออะไรบ้าง?
4. ทำไมถึงตัดสินใจแบบนี้?

ตอบเป็น JSON ในรูปแบบ:
{{
    "thinking_steps": ["ขั้นตอนการคิด 1", "ขั้นตอนการคิด 2", ...],
    "decision": {{
        "action": "go_shopping" | "stay_home" | "other",
        "store": "ชื่อร้าน",
        "items_to_buy": ["สินค้า 1", "สินค้า 2", ...],
        "estimated_spend": จำนวนเงิน,
        "reasoning": "เหตุผลโดยรวม"
    }}
}}"""
    return prompt


def call_llm(prompt: str) -> Dict:
    """เรียก LLM เพื่อตัดสินใจ

    ถ้า USE_REAL_LLM = True: เรียก OpenAI API จริง
    ถ้า USE_REAL_LLM = False: ใช้ MOCK_DECISION ที่เตรียมไว้
    """
    if not USE_REAL_LLM:
        print("  [โหมด Mock] ใช้ข้อมูลจำลองแทน LLM จริง")
        return MOCK_DECISION

    print("  [โหมด Real LLM] กำลังเรียก OpenAI API...")
    try:
        api_key = os.environ.get("OPENAI_API_KEY")
        if not api_key:
            env_paths = [
                os.path.join(os.getcwd(), "..", "..", ".env"),
                os.path.join(os.getcwd(), ".env"),
            ]
            for env_path in env_paths:
                try:
                    with open(os.path.abspath(env_path), "r") as f:
                        for line in f:
                            line = line.strip()
                            if line.startswith("OPENAI_API_KEY="):
                                api_key = line.split("=", 1)[1].strip().strip('"').strip("'")
                                break
                except FileNotFoundError:
                    continue

        if not api_key:
            print("  ไม่พบ OPENAI_API_KEY! กลับไปใช้โหมด Mock")
            return MOCK_DECISION

        from openai import OpenAI
        client = OpenAI(api_key=api_key)
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "คุณเป็น AI ที่จำลองพฤติกรรมลูกค้า ตอบเป็น JSON เท่านั้น"},
                {"role": "user", "content": prompt}
            ],
            temperature=0.7,
            response_format={"type": "json_object"}
        )
        result_text = response.choices[0].message.content
        result = json.loads(result_text)
        print("  ได้รับคำตอบจาก LLM แล้ว!")
        return result

    except ImportError:
        print("  ไม่พบ library 'openai'! กรุณา: pip install openai")
        print("  กลับไปใช้โหมด Mock")
        return MOCK_DECISION
    except Exception as e:
        print(f"  เกิดข้อผิดพลาดในการเรียก LLM: {e}")
        print("  กลับไปใช้โหมด Mock")
        return MOCK_DECISION


def call_llm_reflect(experience: str) -> Dict:
    """เรียก LLM เพื่อสะท้อนบทเรียนจากประสบการณ์

    ใน step REFLECT agent จะสรุปสิ่งที่เรียนรู้
    ผลลัพธ์จะถูกเก็บเป็นความทรงจำใหม่ใน Memory Stream
    """
    if not USE_REAL_LLM:
        return MOCK_REFLECTION

    try:
        api_key = os.environ.get("OPENAI_API_KEY")
        if not api_key:
            return MOCK_REFLECTION

        from openai import OpenAI
        client = OpenAI(api_key=api_key)

        prompt = f"""จากประสบการณ์ต่อไปนี้ กรุณาสะท้อนบทเรียนที่ได้:

{experience}

ตอบเป็น JSON:
{{
    "reflection": "บทสะท้อนจากประสบการณ์",
    "importance": 1-10,
    "new_memory": "ความทรงจำใหม่ที่จะเก็บ (สั้นกระชับ)"
}}"""

        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "คุณเป็น AI ที่สะท้อนบทเรียนจากประสบการณ์ ตอบเป็น JSON"},
                {"role": "user", "content": prompt}
            ],
            temperature=0.7,
            response_format={"type": "json_object"}
        )
        return json.loads(response.choices[0].message.content)

    except Exception:
        return MOCK_REFLECTION


print("Mock data + LLM functions loaded.")

## Step 1: PERCEIVE (รับรู้)

Agent "มองดู" สิ่งรอบตัว -- **ไม่ต้องใช้ LLM** เพราะเป็นการอ่านข้อมูลจาก simulation environment

สิ่งที่ Agent รับรู้ได้:
1. **สถานะของบ้าน** (inventory) -- ของเหลือเท่าไหร่ อะไรใกล้หมด
2. **ร้านค้าใกล้เคียง** -- ร้านอะไรบ้าง ไกลแค่ไหน
3. **โปรโมชั่นที่มีอยู่** -- ร้านไหนมีโปรอะไร
4. **วัน/เวลาปัจจุบัน** -- เป็นวันหยุดหรือเปล่า

In [ ]:
# ============================================================
# Step 1: PERCEIVE (รับรู้)
# ============================================================
print_step_header(1, "PERCEIVE", "สมศรี สังเกตสิ่งรอบตัว", uses_llm=False)

# แสดง inventory
print("\n  สถานะของในบ้าน (Inventory):")
for item_name, item_data in household.get("inventory", {}).items():
    current = item_data["current"]
    threshold = item_data["threshold"]
    unit = item_data["unit"]
    if current <= threshold:
        status = "  !! ต่ำกว่า threshold! ต้องซื้อ"
    elif current <= threshold * 1.5:
        status = "  ~  ใกล้หมด"
    else:
        status = "  OK ยังพอ"

    bar_val = min(current / (threshold * 3), 1.0)
    print(f"    {item_name:20s}: {current:5.1f} {unit:8s} "
          f"(threshold: {threshold}) {make_bar(bar_val, 15)}{status}")

# แสดง shopping list
print(f"\n  รายการที่ต้องซื้อ (Shopping List):")
for item in household.get("shopping_list", []):
    print(f"    [{item['urgency']:6s}] {item['item']} - {item['reason']}")

# แสดงร้านค้าและโปร
print(f"\n  ร้านค้าใกล้เคียงและโปรโมชั่น:")
for store in environment.get("nearby_stores", []):
    print(f"    {store['name']} ({store['distance_km']} km)")
    if store["current_promotions"]:
        for promo in store["current_promotions"]:
            print(f"      * {promo}")
    else:
        print(f"      (ไม่มีโปรโมชั่น)")

# แสดงสถานการณ์
print(f"\n  สถานการณ์:")
print(f"    วัน: {environment['current_day']}  เวลา: {environment['current_time']}")
print(f"    งบเหลือ: {household['budget_remaining']:,} บาท / {household['monthly_budget']:,} บาท")
print(f"    ซื้อของล่าสุด: {household['last_shopping_date']}")

## Step 2: RETRIEVE (ดึงความทรงจำ)

Agent ดึงความทรงจำที่เกี่ยวข้องมาใช้ประกอบการตัดสินใจ -- **ไม่ต้องใช้ LLM**

ใช้สูตร: `score = recency x importance x relevance`

- **recency** = 0.99 ^ hours_since_event (ยิ่งเร็วยิ่งสูง)
- **importance** = 1-10 (LLM ให้คะแนนตอนสร้างความทรงจำ)
- **relevance** = cosine similarity กับ query

Query ที่ใช้ค้นหา: *"ซื้อของ นม ไข่ โปรโมชั่น ร้าน"*

In [ ]:
# ============================================================
# Step 2: RETRIEVE (ดึงความทรงจำ)
# ============================================================
print_step_header(2, "RETRIEVE", "ดึงความทรงจำที่เกี่ยวข้องจาก Memory Stream", uses_llm=False)

memories = get_relevant_memories()

print(f"\n  ความทรงจำ Top-5 ที่ดึงมาได้:\n")
for i, mem in enumerate(memories, 1):
    print(f"    #{i} [{mem['type']}] score={mem['total_score']:.3f}")
    print(f"       \"{mem['description']}\"")
    print(f"       recency={mem['recency']:.3f} x "
          f"importance={mem['importance']:.3f} x "
          f"relevance={mem['relevance']:.3f} = {mem['total_score']:.3f}"
          f"  ({mem['hours_ago']} ชม. ก่อน)")
    print(f"       {make_bar(mem['total_score'])}")
    print()

## Step 3: PLAN (วางแผน) -- [ใช้ LLM]

นี่คือ **ขั้นตอนที่สำคัญที่สุด!** Agent ใช้ LLM ตัดสินใจ

Agent จะส่ง prompt ที่รวมข้อมูลทั้งหมด:
- ตัวตน + บุคลิก + ประวัติ
- สถานการณ์ปัจจุบัน (inventory, budget, วัน/เวลา)
- ความทรงจำที่เกี่ยวข้อง
- ร้านค้า + โปรโมชั่น

LLM จะคิดแบบ **step-by-step** (Chain-of-Thought) แล้วตัดสินใจ

> **หลักการ: ทุกการตัดสินใจผ่าน LLM ไม่ใช่ if-else!**
> นี่คือสิ่งที่ทำให้ Generative Agents แตกต่างจาก rule-based simulation

In [ ]:
# ============================================================
# Step 3: PLAN (วางแผน - ใช้ LLM!)
# ============================================================
print_step_header(3, "PLAN", "ใช้ LLM คิดว่าจะทำอะไร", uses_llm=True)

# สร้าง prompt
prompt = build_llm_prompt(customer, household, environment, memories)

# แสดง prompt (ย่อ)
print(f"\n  Prompt ที่ส่งให้ LLM (แสดงบางส่วน):")
print(f"  {'─' * 56}")

prompt_lines = prompt.split('\n')
for line in prompt_lines[:15]:
    print(f"    {line}")
if len(prompt_lines) > 15:
    print(f"    ... (อีก {len(prompt_lines) - 15} บรรทัด)")
    for line in prompt_lines[-12:]:
        print(f"    {line}")
print(f"  {'─' * 56}")

print(f"\n  ความยาว prompt ทั้งหมด: {len(prompt)} ตัวอักษร, {len(prompt_lines)} บรรทัด")

# เรียก LLM
print(f"\n  กำลังส่งให้ LLM คิด...")
decision = call_llm(prompt)

# แสดงกระบวนการคิด (thinking_steps)
print(f"\n  กระบวนการคิดของ Agent (Chain-of-Thought):\n")
for i, step in enumerate(decision.get("thinking_steps", []), 1):
    print(f"    {i}. {step}")
print()

# แสดงการตัดสินใจ
d = decision.get("decision", {})
print_box("การตัดสินใจของ สมศรี", f"""
  Action:  {d.get('action', 'N/A')}
  Store:   {d.get('store', 'N/A')}
  Items:   {', '.join(d.get('items_to_buy', []))}
  Budget:  ~{d.get('estimated_spend', 'N/A')} บาท
  Reason:  {d.get('reasoning', 'N/A')}
""")

## Step 4: EXECUTE (ลงมือทำ)

Agent ลงมือทำตามที่ตัดสินใจ -- **ไม่ต้องใช้ LLM** เพราะเป็นการ update สถานะใน simulation

ขั้นตอนนี้จะ:
- เปลี่ยนตำแหน่งของ Agent (เดินทางไปร้าน)
- ซื้อสินค้าตามรายการ
- Update inventory, budget, shopping list

In [ ]:
# ============================================================
# Step 4: EXECUTE (ลงมือทำ)
# ============================================================
print_step_header(4, "EXECUTE", "ลงมือทำตามแผน", uses_llm=False)

store_name = d.get("store", "CJ More")
items = d.get("items_to_buy", ["นม", "ไข่", "ผงซักฟอก"])
spend = d.get("estimated_spend", 350)

print(f"\n  กำลังดำเนินการ:\n")

# แสดงการดำเนินการแบบ step-by-step
actions = [
    f"สมศรี ออกจากบ้านไปที่ {store_name}...",
    f"ถึง {store_name} แล้ว กำลังเดินเลือกซื้อของ...",
]
for item in items:
    actions.append(f"  หยิบ {item} ใส่ตะกร้า")
actions.extend([
    f"เดินไปจ่ายเงินที่เคาน์เตอร์...",
    f"จ่ายเงินทั้งหมด ~{spend} บาท",
    f"ขับรถกลับบ้าน เก็บของเข้าที่"
])

for action in actions:
    print(f"    > {action}")

# แสดงการ update inventory
print(f"\n  Update สถานะ (ใน simulation):")
print(f"    budget_remaining: {household['budget_remaining']:,} --> "
      f"{household['budget_remaining'] - spend:,} บาท")
print(f"    last_shopping_date: {household['last_shopping_date']} --> 2024-01-13")
print(f"    shopping_list: {len(household.get('shopping_list', []))} items --> "
      f"0 items (ซื้อครบแล้ว)")

## Step 5: REFLECT (สะท้อน) -- [ใช้ LLM]

ขั้นตอนสุดท้าย: Agent สะท้อนบทเรียนจากสิ่งที่เพิ่งทำ

LLM จะสรุป:
- สิ่งที่เรียนรู้จากประสบการณ์
- ให้คะแนนความสำคัญ (importance score)
- สร้างความทรงจำใหม่เก็บใน Memory Stream

**ทำไมต้อง Reflect?**
- เพื่อให้ agent "เรียนรู้" จากประสบการณ์
- ครั้งหน้าเมื่อต้องตัดสินใจ ความทรงจำนี้จะถูกดึงมาใช้
- ทำให้พฤติกรรม agent **เปลี่ยนแปลงตามเวลา** (เหมือนคนจริง!)

In [ ]:
# ============================================================
# Step 5: REFLECT (สะท้อน - ใช้ LLM!)
# ============================================================
print_step_header(5, "REFLECT", "สร้างความทรงจำใหม่จากประสบการณ์", uses_llm=True)

# สรุปประสบการณ์ที่จะส่งให้ LLM
experience = (
    f"สมศรี ไปซื้อของที่ {store_name} วันเสาร์เช้า "
    f"ซื้อ {', '.join(items)} "
    f"ใช้เงินประมาณ {spend} บาท "
    f"ได้โปรโมชั่นนมลด 30% ไข่ซื้อ 2 แถม 1 ผงซักฟอกลด 25% "
    f"บริการดี พนักงานสุภาพ จอดรถสะดวก"
)

print(f"\n  ประสบการณ์ที่ส่งให้ LLM สะท้อน:")
print(f"    \"{experience}\"")

print(f"\n  กำลังให้ LLM สะท้อนบทเรียน...")
reflection = call_llm_reflect(experience)

# แสดงผลการสะท้อน
print(f"\n  บทสะท้อนจาก Agent:")
print_box("Reflection (บทสะท้อน)", f"""
  "{reflection.get('reflection', 'N/A')}"

  Importance Score: {reflection.get('importance', 'N/A')}/10

  New Memory (เก็บใน Memory Stream):
  "{reflection.get('new_memory', 'N/A')}"
""")

print(f"\n  ความทรงจำใหม่นี้จะถูกเก็บใน Memory Stream")
print(f"  ครั้งหน้าเมื่อสมศรีต้องตัดสินใจซื้อของ")
print(f"  ความทรงจำนี้จะถูกดึงมาใช้ด้วย (ถ้า query เกี่ยวข้อง)")

## สรุป: Cognitive Loop ครบ 1 รอบ

```
1. PERCEIVE  --> สมศรีเห็นว่านมไข่ใกล้หมด เห็นโปรร้านต่างๆ
2. RETRIEVE  --> ดึงความทรงจำ: โปร CJ More ดี, Big C จอดรถยาก
3. PLAN      --> LLM ตัดสินใจ: ไป CJ More เพราะโปรดี+บริการดี
4. EXECUTE   --> ไปซื้อนม ไข่ ผงซักฟอก จ่าย ~350 บาท
5. REFLECT   --> สรุป: CJ More คุ้ม ควรมาอีก (เก็บเป็นความทรงจำใหม่)
```

### จุดสำคัญที่ควรเข้าใจ

**1. LLM ตัดสินใจ ไม่ใช่ if-else**
- ไม่มี "ถ้าโปรลด > 20% ก็ไป" เป็น code
- LLM คิดเอง พิจารณาหลายปัจจัยพร้อมกัน

**2. Personality ส่งผลต่อการตัดสินใจ**
- สมศรี price_sensitivity=0.8 --> LLM จะเลือกร้านราคาดี
- ถ้าเป็น agent อื่น (convenience=0.9) อาจเลือก 7-Eleven

**3. Memory ทำให้ agent "เรียนรู้"**
- ครั้งก่อน Big C จอดรถยาก --> ครั้งนี้ไม่ไป Big C
- ครั้งก่อน CJ More บริการดี --> ครั้งนี้เลือก CJ More

**4. Reflection สร้างความทรงจำระยะยาว**
- ประสบการณ์ดี --> จำไว้ --> ครั้งหน้ากลับมาอีก
- ประสบการณ์แย่ --> จำไว้ --> ครั้งหน้าหลีกเลี่ยง

> **นี่คือสิ่งที่ทำให้ Generative Agents สมจริงกว่า rule-based simulation!**